# MPT Agent System - Gemini Colab Runner
**Runtime: Standard (CPU)** - No GPU needed.

Pipeline: Trends -> Script (Gemini) -> Video (MPT stock footage API) -> SEO (Gemini) -> Publish

## Step 1 - Clone repos and install dependencies

In [ ]:
# Clone MoneyPrinterTurbo
!git clone https://github.com/harry0703/MoneyPrinterTurbo.git /content/MoneyPrinterTurbo 2>/dev/null || (cd /content/MoneyPrinterTurbo && git pull)

# Install MPT dependencies (uses pyproject.toml)
!cd /content/MoneyPrinterTurbo && pip install -q -e . 2>/dev/null || pip install -q -r requirements.txt 2>/dev/null || echo 'MPT deps: using fallback'
!pip install -q fastapi uvicorn edge-tts moviepy imageio-ffmpeg

# Clone agent system
!git clone https://github.com/Bhushan-Khachane/mpt-agent-system.git /content/mpt-agent-system 2>/dev/null || (cd /content/mpt-agent-system && git pull)

# Install agent dependencies
!pip install -q feedparser pytrends requests google-api-python-client google-auth-httplib2 google-auth-oauthlib

print('All dependencies installed.')

## Step 2 - Paste your keys HERE

**Gemini key (free):** https://aistudio.google.com/app/apikey

**Pexels key (free):** https://www.pexels.com/api/

In [ ]:
import os, sys, pathlib

# ---- EDIT THESE LINES ----
GEMINI_API_KEY  = 'AIza...'          # <-- paste your real Gemini key
PEXELS_API_KEY  = 'YOUR_PEXELS_KEY'  # <-- paste your Pexels key
PIXABAY_API_KEY = 'YOUR_PIXABAY_KEY' # <-- optional
# --------------------------

if not GEMINI_API_KEY or GEMINI_API_KEY in ('AIza...', 'YOUR_GEMINI_API_KEY', ''):
    raise ValueError('ERROR: Paste your real Gemini key above. Get it free at https://aistudio.google.com/app/apikey')

os.environ['GEMINI_API_KEY']  = GEMINI_API_KEY
os.environ['GEMINI_MODEL']    = 'gemini-2.0-flash'
os.environ['PEXELS_API_KEY']  = PEXELS_API_KEY
os.environ['PIXABAY_API_KEY'] = PIXABAY_API_KEY
os.environ['MPT_DIR']         = '/content/MoneyPrinterTurbo'
os.environ['MPT_USE_API']     = 'true'
os.environ['MPT_API_URL']     = 'http://127.0.0.1:8080'
os.environ['YT_CHANNEL_AI']    = 'UCxxxxxxxxxxxxxxxx'
os.environ['YT_CHANNEL_CLEAN'] = 'UCxxxxxxxxxxxxxxxx'
os.environ['YT_CHANNEL_MONEY'] = 'UCxxxxxxxxxxxxxxxx'
os.environ['IG_ACCOUNT_AI']    = 'your_ig_handle'
os.environ['IG_ACCOUNT_CLEAN'] = 'your_ig_handle'
os.environ['IG_ACCOUNT_MONEY'] = 'your_ig_handle'

sys.path.insert(0, '/content/mpt-agent-system')
for d in ['data','videos','logs','credentials']:
    pathlib.Path(f'/content/mpt-agent-system/{d}').mkdir(parents=True, exist_ok=True)
for fp, val in [
    ('/content/mpt-agent-system/data/topics_queue.json', '[]'),
    ('/content/mpt-agent-system/data/seen_topics.json',  '[]'),
]:
    p = pathlib.Path(fp)
    if not p.exists(): p.write_text(val)

print(f'Config OK. Model={os.environ["GEMINI_MODEL"]} Key={GEMINI_API_KEY[:10]}...')

## Step 3 - TrendScout (RSS + curated fallback topics)

In [ ]:
import os
os.chdir('/content/mpt-agent-system')  # MUST be set before import
from agents.trend_scout import run_trend_scout
topics = run_trend_scout()
print(f'\nTop 5 topics queued:')
for t in topics[:5]:
    print(f"  [{t['niche']}] {t['title']} (score={t['score_val']:.1f}, src={t['source']})")

## Step 4 - ScriptWriter (Gemini API)
> Scripts are generated by Gemini 2.0 Flash. Each script ~50-80 words.

In [ ]:
import os, json
os.chdir('/content/mpt-agent-system')
from agents.script_writer import run_script_writer
run_script_writer(batch_size=6)

queue    = json.loads(open('data/topics_queue.json').read())
scripted = [t for t in queue if t.get('status') == 'scripted']
print(f'\n{len(scripted)} scripts ready')
if scripted:
    print(f"--- Sample ({scripted[0]['niche']}) ---")
    print(scripted[0]['script'])

## Step 5 - VideoProducer (MPT stock footage)
> Starts MPT as a background API server, then submits each script as a video task.
> Uses Pexels stock footage only. Portrait mode (9:16) for Shorts/Reels.

In [ ]:
import os
os.chdir('/content/mpt-agent-system')
from agents.video_producer import run_video_producer
# use_api=True: starts MPT as background server (correct for Colab)
run_video_producer(batch_size=3, use_api=True)

## Step 6 - SEO Agent (Gemini API)

In [ ]:
import os
os.chdir('/content/mpt-agent-system')
from agents.seo_agent import run_seo_agent
run_seo_agent(batch_size=6)

## Step 7 - Publisher (YouTube + Instagram + Facebook)
> Upload OAuth token files to /content/mpt-agent-system/credentials/ before running.

In [ ]:
import os
os.chdir('/content/mpt-agent-system')
from agents.publisher import run_publisher
run_publisher(batch_size=3)

## One-shot: run full pipeline (Steps 3-7)

In [ ]:
import os
os.chdir('/content/mpt-agent-system')
!python orchestrator.py